# Solution 05 — streaming + anomalies

In [ ]:
from confluent_kafka import Producer
from datetime import datetime
import json, time, random

BROKER = 'redpanda:29092'

producer = Producer({
    'bootstrap.servers': BROKER,
    'client.id':         'streaming-producer',
})

riders = ['rider_3', 'rider_5', 'rider_6', 'rider_7']
NORMAL_MAX_KMH = 35.0   # a bike or scooter in town does not go faster

# Every rider starts somewhere near Lucerne main station.
positions = {r: [47.0502 + random.uniform(-0.01, 0.01),
                 8.3093 + random.uniform(-0.01, 0.01)] for r in riders}


def move(rider):
    """Move the rider a few metres and return (lat, lon, speed_kmh)."""
    positions[rider][0] += random.uniform(-0.0003, 0.0003)
    positions[rider][1] += random.uniform(-0.0003, 0.0003)
    speed = round(random.uniform(0.0, NORMAL_MAX_KMH), 1)   # 0 = waiting at a restaurant
    return round(positions[rider][0], 5), round(positions[rider][1], 5), speed

print('Ready.')

## Step 1 — continuous stream

Notice we `flush()` after every message here so the consumer sees events one-by-one instead of in batches. That's nice for the demo but not what you'd do in production — let librdkafka batch.

In [ ]:
MAX_EVENTS = 250
count = 0
print(f'{"Time":>10} | {"#":>4} | {"Rider":>7} | {"Lat":>9} | {"Lon":>8} | {"km/h":>5}')
print('-' * 60)

try:
    while count < MAX_EVENTS:
        rider           = random.choice(riders)
        lat, lon, speed = move(rider)
        event = json.dumps({
            'rider_id':  rider,
            'lat':       lat,
            'lon':       lon,
            'speed_kmh': speed,
            'timestamp': time.time(),
        })
        producer.produce('rider-locations', key=rider.encode(), value=event.encode())
        producer.flush()
        count += 1
        ts = datetime.now().strftime('%H:%M:%S')
        print(f'{ts:>10} | {count:>4} | {rider:>7} | {lat:>9.5f} | {lon:>8.5f} | {speed:>5.1f}')
        time.sleep(random.uniform(0.5, 2.0))
except KeyboardInterrupt:
    producer.flush()
    print(f'\nStopped. Total: {count}')

## Task A — burst (no sleep)

Compare the throughput line in your consumer: a 50-event burst completes here in well under a second. That's Kafka's whole marketing line in action.

In [ ]:
for _ in range(50):
    rider           = random.choice(riders)
    lat, lon, speed = move(rider)
    event = json.dumps({'rider_id': rider, 'lat': lat, 'lon': lon,
                        'speed_kmh': speed, 'timestamp': time.time()})
    producer.produce('rider-locations', key=rider.encode(), value=event.encode())
producer.flush()   # one flush, AFTER the loop
print('Burst sent.')

## Task B — anomaly simulation

We tag the message itself rather than recomputing the threshold downstream. The consumer in exercise 06 will compare the speed against its own limit.

In [ ]:
n_anoms = 0
for i in range(30):
    rider           = random.choice(riders)
    lat, lon, speed = move(rider)
    is_anom = random.random() < 0.2
    if is_anom:
        # GPS glitch: the fix jumps several hundred metres, the computed
        # speed is 3-8x the normal max — clearly impossible on a bike
        lat   = round(lat + random.uniform(-0.01, 0.01), 5)
        lon   = round(lon + random.uniform(-0.01, 0.01), 5)
        speed = round(random.uniform(NORMAL_MAX_KMH * 3, NORMAL_MAX_KMH * 8), 1)
        n_anoms += 1
    event = json.dumps({'rider_id': rider, 'lat': lat, 'lon': lon,
                        'speed_kmh': speed, 'timestamp': time.time(),
                        'anomaly': is_anom})
    producer.produce('rider-locations', key=rider.encode(), value=event.encode())
    producer.flush()
    note = '<-- ANOMALY' if is_anom else ''
    print(f'{i+1:>3} | {rider:>7} | {speed:>6.1f} km/h | {note}')
print(f'\n{n_anoms}/30 events were anomalies.')